# LangCalc Training

<img src="https://raw.githubusercontent.com/jedick/LangCalc/main/assets/langcalc-icon-outline.svg" alt="LangCalc icon" width="100">

Training notebook for the [LangCalc](https://github.com/jedick/LangCalc) voice calculator function-calling model. This notebook is adapted from [Fine-tuning with FunctionGemma](https://github.com/google-gemma/cookbook/blob/main/docs/functiongemma/finetuning-with-functiongemma.ipynb).

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/jedick/LangCalc/blob/main/model/notebooks/LangCalc-training.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://jedick/LangCalc/cookbook/blob/main/model/notebooks/LangCalc-training.ipynb"><img src="https://www.kaggle.com/static/images/logos/kaggle-logo-transparent-300.png" height="32" width="70"/>Run in Kaggle</a>
  </td>
  <td>
    <a target="_blank" href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fjedick%2FLangCalc%2Fmain%2Fmodel%2Fnotebooks%2FLangCalc-training.ipynb"><img src="https://ai.google.dev/images/cloud-icon.svg" width="40" />Open in Vertex AI</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/google-gemma/cookbook/blob/main/model/notebooks/LangCalc-training.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

## Setup development environment

The first step is to install Hugging Face Libraries, including TRL, and datasets to fine-tune open model, including different RLHF and alignment techniques.

In [ ]:
# Install Pytorch & other libraries
%pip install torch

# Install Hugging Face libraries
%pip install transformers datasets trl

# COMMENT IN: if you are running on a GPU that supports BF16 data type and flash attn, such as NVIDIA L4 or NVIDIA A100
#%pip install flash-attn

> _Note: If you are using a GPU with Ampere architecture (such as NVIDIA L4) or newer, you can use Flash attention. Flash Attention is a method that significantly speeds computations up and reduces memory usage from quadratic to linear in sequence length, leading to accelerating training up to 3x. Learn more at [FlashAttention](https://github.com/Dao-AILab/flash-attention/tree/main)._

Before you can start training, you have to make sure that you accepted the terms of use for Gemma. You can accept the license on [Hugging Face](http://huggingface.co/google/functiongemma-270m-it) by clicking on the **Agree** and access repository button on the model page at: http://huggingface.co/google/functiongemma-270m-it

After you have accepted the license, you need a valid Hugging Face Token to access the model. If you are running inside a Google Colab, you can securely use your Hugging Face Token using the Colab secrets otherwise you can set the token as directly in the `login` method. Make sure your token has write access too, as you push your model to Hugging Face Hub after fine-tuning.

In [ ]:
# Retrieve HF_TOKEN from Secrets
from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")

# Login into Hugging Face Hub
from huggingface_hub import login
login(hf_token)

You can keep the results on Colab's local virtual machine. However, it is highly recommended saving your intermediate results to your Google Drive. This ensures your training results are safe and allows you to easily compare and select the best model.

Also, adjust the checkpoint directory and the learning rate.

In [ ]:
from google.colab import drive

mount_google_drive = True #@param {type:"boolean"}
model_name = "functiongemma-270m-ft-langcalc" #@param {type:"string"}
output_dir = model_name

# Paths to the training/test CSVs (place these files in the same Drive folder,
# or update the paths below).
train_csv_path = "LangCalc/train_data.csv"  # @param {type:"string"}
test_csv_path = "LangCalc/test_data.csv"  # @param {type:"string"}

if mount_google_drive:
    drive.mount('/content/drive')
    output_dir = f"/content/drive/MyDrive/{model_name}"
    train_csv_path = f"/content/drive/MyDrive/{train_csv_path}"
    test_csv_path = f"/content/drive/MyDrive/{test_csv_path}"

print(f"Checkpoints will be saved to {output_dir}")
print(f"Train CSV: {train_csv_path}")
print(f"Test CSV: {test_csv_path}")

base_model = "google/functiongemma-270m-it"

## Prepare the fine-tuning dataset


## Tool definitions

In [ ]:
import json
import math
import re

import pandas as pd
from datasets import Dataset
from transformers.utils import get_json_schema

# --- Tool definitions, taken from calc_app.py ---

def add(x: float, y: float):
    """
    Adds two numbers together (sum, total, plus).

    Args:
        x: the first number
        y: the second number

    Returns:
        result: the sum of x and y (x + y).
    """
    return {"result": x + y}


def subtract(x: float, y: float):
    """
    Subtracts one number from another (difference, minus).

    Args:
        x: the starting number
        y: the number to subtract from x

    Returns:
        result: the difference between x and y (x - y).
    """
    return {"result": x - y}


def multiply(x: float, y: float):
    """
    Multiplies two numbers together (product, times).

    Args:
        x: the first number
        y: the second number

    Returns:
        result: the product of x and y (x * y).
    """
    return {"result": x * y}


def divide(x: float, y: float):
    """
    Divides one number by another (quotient, over, go into).

    Args:
        x: the numerator
        y: the denominator

    Returns:
        result: the quotient of x divided by y (x / y).
    """
    return {"result": x / y}


TOOLS = [get_json_schema(t) for t in (add, subtract, multiply, divide)]

DEFAULT_SYSTEM_MSG = "You are a model that can do function calling with the following functions"

## Dataset preparation

In [ ]:
def load_csv_as_dataset(path: str) -> Dataset:
    """Load a train/test CSV (columns: category, prompt, function, x, y)
    into a HF Dataset in the conversational tool-calling format."""

    df = pd.read_csv(path)

    def create_conversation(row):
        return {
            "messages": [
                {"role": "developer", "content": DEFAULT_SYSTEM_MSG},
                {"role": "user", "content": row["prompt"]},
                {
                    "role": "assistant",
                    "tool_calls": [
                        {
                            "type": "function",
                            "function": {
                                "name": row["function"],
                                "arguments": {"x": float(row["x"]), "y": float(row["y"])},
                            },
                        }
                    ],
                },
            ],
            "tools": TOOLS,
            # keep the ground-truth values around (unpacked from tool_calls)
            # for use in check_success_rate()
            "expected_function": row["function"],
            "expected_x": float(row["x"]),
            "expected_y": float(row["y"]),
        }

    records = [create_conversation(row) for _, row in df.iterrows()]
    return Dataset.from_list(records)


dataset = {
    "train": load_csv_as_dataset(train_csv_path),
    "test": load_csv_as_dataset(test_csv_path),
}

print(f"Train examples: {len(dataset['train'])}")
print(f"Test examples: {len(dataset['test'])}")

## Fine-tune FunctionGemma using TRL and the SFTTrainer

You are now ready to fine-tune your model. Hugging Face TRL [SFTTrainer](https://huggingface.co/docs/trl/sft_trainer) makes it straightforward to supervise fine-tune open LLMs. The `SFTTrainer` is a subclass of the `Trainer` from the `transformers` library and supports all the same features,

The following code loads the FunctionGemma model and tokenizer from Hugging Face.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    dtype="auto",
    device_map="auto",
    attn_implementation="eager",
)
tokenizer = AutoTokenizer.from_pretrained(base_model)

print(f"Device: {model.device}")
print(f"DType: {model.dtype}")

# Print formatted user prompt
print("--- dataset input ---")
print(json.dumps(dataset["train"][0], indent=2))
debug_msg = tokenizer.apply_chat_template(
    dataset["train"][0]["messages"],
    tools=dataset["train"][0]["tools"],
    add_generation_prompt=False,
    tokenize=False,
)
print("--- Formatted prompt ---")
print(debug_msg)

## Before fine-tune

The output below shows that the out-of-the-box capabilities may not be good enough for this use case.

In [ ]:
# Unlike the original example (which only checks whether the right tool
# name shows up in the output), our tool arguments (x, y) are numeric
# and have a single deterministic correct value for every prompt. So we
# run two independent checks per example:
#   1. correct function name
#   2. correct arguments (x and y)
# Both must pass for the example to count as correct.

def extract_tool_calls(text):
    """Parses FunctionGemma's <start_function_call> ... <end_function_call>
    output into a list of {"name": ..., "arguments": {...}} dicts.
    (Same parsing logic as calc_app.py's extract_tool_calls.)"""

    def cast(v):
        try:
            return int(v)
        except ValueError:
            try:
                return float(v)
            except ValueError:
                return {"true": True, "false": False}.get(v.lower(), v.strip("'\""))

    return [
        {
            "name": name,
            "arguments": {
                k: cast((v1 or v2).strip())
                for k, v1, v2 in re.findall(r"(\w+):(?:<escape>(.*?)<escape>|([^,}]*))", args)
            },
        }
        for name, args in re.findall(
            r"<start_function_call>call:(\w+)\{(.*?)\}<end_function_call>", text, re.DOTALL
        )
    ]


def numbers_match(actual, expected, tol=1e-6):
    if actual is None:
        return False
    try:
        return math.isclose(float(actual), float(expected), rel_tol=tol, abs_tol=tol)
    except (TypeError, ValueError):
        return False


def check_success_rate():
    success_count = 0
    for idx, item in enumerate(dataset["test"]):
        messages = [
            item["messages"][0],
            item["messages"][1],
        ]

        inputs = tokenizer.apply_chat_template(
            messages, tools=TOOLS, add_generation_prompt=True, return_dict=True, return_tensors="pt"
        )

        out = model.generate(
            **inputs.to(model.device), pad_token_id=tokenizer.eos_token_id, max_new_tokens=128
        )
        output = tokenizer.decode(out[0][len(inputs["input_ids"][0]):], skip_special_tokens=False)

        expected_function = item["expected_function"]
        expected_x = item["expected_x"]
        expected_y = item["expected_y"]

        calls = extract_tool_calls(output)
        first_call = calls[0] if calls else None

        actual_function = first_call["name"] if first_call else None
        actual_x = first_call["arguments"].get("x") if first_call else None
        actual_y = first_call["arguments"].get("y") if first_call else None

        func_ok = actual_function == expected_function
        args_ok = numbers_match(actual_x, expected_x) and numbers_match(actual_y, expected_y)

        func_symbol = "✅" if func_ok else "❌"
        args_symbol = "✅" if args_ok else "❌"

        if func_ok and args_ok:
            #print("  -> ✅ correct!")
            success_count += 1
        else:
            print(f"{idx + 1} Prompt: {item['messages'][1]['content']}")
            #print(f"  Output: {output}")
            print(
                f"  Function: {func_symbol} (expected '{expected_function}', got "
                f"{repr(actual_function)})"
            )
            print(
                f"  Arguments: {args_symbol} (expected x={expected_x}, y={expected_y}, got "
                f"x={actual_x}, y={actual_y})"
            )
            #print(f"  -> {func_symbol}{args_symbol} failed")

    print(f"Success : {success_count} / {len(dataset['test'])}")

In [ ]:
check_success_rate()

## Training

Before you can start your training, you need to define the hyperparameters you want to use in a `SFTConfig` instance.

In [ ]:
from trl import SFTConfig

torch_dtype = model.dtype

args = SFTConfig(
    output_dir=output_dir,                  # directory to save model checkpoints
    max_length=512,                         # max sequence length for model and packing of the dataset
    packing=False,                          # Groups multiple samples in the dataset into a single sequence
    num_train_epochs=4,                     # number of training epochs
    per_device_train_batch_size=8,          # batch size per device during training
    train_sampling_strategy="random",       # shuffle each epoch (this is the default)
    gradient_checkpointing=False,           # Caching is incompatible with gradient checkpointing
    optim="adamw_torch_fused",              # use fused adamw optimizer
    logging_steps=1,                        # log every step
    save_strategy="no",                     # don't save checkpoint every epoch
    eval_strategy="epoch",                  # evaluate checkpoint every epoch
    learning_rate=5e-5,                     # learning rate
    fp16=True if torch_dtype == torch.float16 else False,   # use float16 precision
    bf16=True if torch_dtype == torch.bfloat16 else False,  # use bfloat16 precision
    lr_scheduler_type="constant",            # use constant learning rate scheduler
)

You now have every building block you need to create your `SFTTrainer` to start the training of your model.

In [ ]:
from trl import SFTTrainer

# Create Trainer object
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    processing_class=tokenizer,
)

Start training by calling the `train()` method.

In [ ]:
# Start training
trainer.train()

To plot the training and validation losses, you would typically extract these values from the `TrainerState` object or the logs generated during training.

Libraries like Matplotlib can then be used to visualize these values over training steps or epochs. The x-axis would represent the training steps or epochs, and the y-axis would represent the corresponding loss values.

In [ ]:
import matplotlib.pyplot as plt

# Access the log history
log_history = trainer.state.log_history

# Extract training / validation loss
train_losses = [log["loss"] for log in log_history if "loss" in log]
epoch_train = [log["epoch"] for log in log_history if "loss" in log]
eval_losses = [log["eval_loss"] for log in log_history if "eval_loss" in log]
epoch_eval = [log["epoch"] for log in log_history if "eval_loss" in log]

# Plot the training loss
plt.plot(epoch_train, train_losses, label="Training Loss")
plt.plot(epoch_eval, eval_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss per Epoch")
plt.legend()
plt.grid(True)
plt.show()

## Test Model Inference

After the training is done, you'll want to evaluate and test your model. You can load different samples from the test dataset and evaluate the model on those samples.


In [ ]:
check_success_rate()

## Optional: Upload the model to Hugging Face Hub

If you're satisfied with the model performance, upload it to Hugging Face Hub so you easily share your model or access it later (for example, in a separate inference notebook or Gradio app).

In [ ]:
from huggingface_hub import ModelCard, whoami

# # Save the final model to output_dir
# trainer.save_model()
# model = AutoModelForCausalLM.from_pretrained(output_dir, device_map="auto")
# tokenizer = AutoTokenizer.from_pretrained(output_dir)

username = whoami()['name']
hf_repo_id = f"{username}/{model_name}"

repo_url = model.push_to_hub(hf_repo_id, commit_message="Upload model")
tokenizer.push_to_hub(hf_repo_id)

card_content = f"""
---
base_model: {base_model}
tags:
- function-calling
- gemma
---
A fine-tuned model based on `{base_model}`."""
card = ModelCard(card_content)

card.push_to_hub(hf_repo_id)

print(f"Uploaded to {repo_url}")